# Crop image classifier

Train an image classifier that predicts whether a photo shows Wheat, Paddy, Mustard, Maize, Sugarcane, or Cotton. This notebook trains a new classification head using MobileNetV2 transfer learning and your labelled images; it does not contain pre-trained weights for these six crops.

Use Python 3.10–3.12 with a working internet connection for the first dependency and ImageNet-weight downloads. Put your images into this structure before training:

```text
crop_dataset/
  Wheat/       # wheat crop images only
  Paddy/       # paddy/rice crop images only
  Mustard/
  Maize/
  Sugarcane/
  Cotton/
```

Aim for at least 100 varied, correctly labelled images per class. More variation in lighting, backgrounds, growth stages, and camera angles generally improves results. Keep unrelated images out of these folders. The notebook holds out test images so evaluation is on images not used for training.

In [3]:
%pip install "tensorflow>=2.15" scikit-learn matplotlib seaborn pillow matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow>=2.15 (from versions: none)
ERROR: No matching distribution found for tensorflow>=2.15


In [4]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ["Wheat", "Paddy", "Mustard", "Maize", "Sugarcane", "Cotton"]
DATASET_DIR = Path("crop_dataset")
OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("Dataset:", DATASET_DIR.resolve())

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png"}
missing_folders = [name for name in CLASS_NAMES if not (DATASET_DIR / name).is_dir()]
if missing_folders:
    raise FileNotFoundError(
        f"Missing class folders in {DATASET_DIR.resolve()}: {missing_folders}. "
        "Create the folder structure described above and add labelled images."
    )

records = []
class_counts = {}
for label in CLASS_NAMES:
    files = sorted(
        path for path in (DATASET_DIR / label).rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    class_counts[label] = len(files)
    records.extend({"path": str(path), "label": label} for path in files)

display(pd.Series(class_counts, name="image_count").to_frame())
too_small = {name: count for name, count in class_counts.items() if count < 10}
if too_small:
    raise ValueError(
        f"At least 10 images per class are needed to create stratified splits: {too_small}. "
        "About 100+ varied images per class is recommended for useful results."
    )

data = pd.DataFrame(records)
data.head()

In [ ]:
# Split by image path before decoding: 70% train, 15% validation, 15% test.
train_df, held_out_df = train_test_split(
    data, test_size=0.30, random_state=SEED, stratify=data["label"]
)
validation_df, test_df = train_test_split(
    held_out_df, test_size=0.50, random_state=SEED, stratify=held_out_df["label"]
)

for split_name, split_df in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    print(f"{split_name}: {len(split_df)} images")
    print(split_df["label"].value_counts().reindex(CLASS_NAMES).to_dict())

label_to_id = {name: index for index, name in enumerate(CLASS_NAMES)}
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(frame, training=False):
    paths = frame["path"].to_numpy()
    labels = frame["label"].map(label_to_id).to_numpy(dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        contents = tf.io.read_file(path)
        image = tf.io.decode_image(contents, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE)
        return image, label

    dataset = dataset.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
validation_ds = make_dataset(validation_df)
test_ds = make_dataset(test_df)

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

base_model = MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet", pooling="avg"
)
base_model.trainable = False

inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 127.5, offset=-1, name="mobilenet_preprocessing")(x)
x = base_model(x, training=False)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="crop_probabilities")(x)
model = keras.Model(inputs, outputs, name="crop_image_classifier")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
best_model_path = OUTPUT_DIR / "crop_classifier_best.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(
        best_model_path, monitor="val_accuracy", save_best_only=True, mode="max"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6
    ),
]

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=20,
    callbacks=callbacks,
)

In [ ]:
# Fine-tune the final feature layers using a small learning rate.
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
fine_tune_history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    callbacks=callbacks,
)

In [ ]:
# Evaluate on held-out test images, which were not used for model fitting.
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
probabilities = model.predict(test_ds, verbose=0)
y_true = test_df["label"].map(label_to_id).to_numpy()
y_pred = probabilities.argmax(axis=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")
print(classification_report(y_true, y_pred, labels=range(len(CLASS_NAMES)), target_names=CLASS_NAMES, zero_division=0))

matrix = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted crop")
plt.ylabel("Actual crop")
plt.title("Held-out test confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model and its exact output-label order.
model_path = OUTPUT_DIR / "crop_classifier.keras"
labels_path = OUTPUT_DIR / "class_names.json"
model.save(model_path)
labels_path.write_text(json.dumps(CLASS_NAMES, indent=2), encoding="utf-8")
print(f"Saved model: {model_path.resolve()}")
print(f"Saved labels: {labels_path.resolve()}")

In [ ]:
def predict_crop(image_path, expected_crop=None, confidence_threshold=0.60):
    """Predict a crop and optionally check it against the booking's selected crop."""
    if expected_crop is not None and expected_crop not in CLASS_NAMES:
        raise ValueError(f"expected_crop must be one of {CLASS_NAMES}")
    if not 0 <= confidence_threshold <= 1:
        raise ValueError("confidence_threshold must be between 0 and 1")

    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path.resolve()}")

    with Image.open(image_path) as source:
        image = source.convert("RGB").resize(IMAGE_SIZE)
        pixels = np.asarray(image, dtype=np.float32)

    scores = model.predict(np.expand_dims(pixels, axis=0), verbose=0)[0]
    predicted_index = int(np.argmax(scores))
    predicted_crop = CLASS_NAMES[predicted_index]
    confidence = float(scores[predicted_index])
    result = {
        "predicted_crop": predicted_crop,
        "confidence": confidence,
        "expected_crop": expected_crop,
    }

    if confidence < confidence_threshold:
        result["status"] = "uncertain"
    elif expected_crop is None:
        result["status"] = "prediction_only"
    else:
        result["status"] = "match" if predicted_crop == expected_crop else "mismatch"
    return result

# Example after training:
# predict_crop("path/to/crop-photo.jpg", expected_crop="Wheat")

In [ ]:
# Set these to a real image and the crop name selected in the booking form.
SAMPLE_IMAGE = None  # Example: "crop_dataset/Wheat/example.jpg"
EXPECTED_CROP = "Wheat"

if SAMPLE_IMAGE and Path(SAMPLE_IMAGE).is_file():
    print(predict_crop(SAMPLE_IMAGE, expected_crop=EXPECTED_CROP))
else:
    print("Skipped: set SAMPLE_IMAGE to a real image path after training.")